# Music Annotation Project — Notebook 34
## V8.2e Registration-Ambiguity Forced-Preservation Visual-Impact Audit

**Runtime:** CPU is sufficient.

This is a post-review, development-only **counterfactual rendering audit**. It does **not** retrain, refit, recalibrate, alter V8.2 probabilities, relabel the source experiment, or authorise production use.

The notebook asks:

> What visual effect results if selected distance-two pixels are treated operationally as PRINT and their exact observed bytes are restored?

Every variant starts from the completed V8.2 page and adds forced preservation at explicitly declared coordinates. It preserves the source V8.2 output everywhere else. The registered V8.2 page remains the baseline and file of record.

### Important interpretation

The reviewed class `Registration ambiguity` is a human development judgement made using observed and clean crops. Forcing those pixels to PRINT is a hypothesis test, not a truth correction. The existing V8.2 truth still calls these distance-two pixels ANNOTATION, so clean-truth safety metrics will count forced pixels as false annotation under the existing rule. Visual review is therefore the primary evidence for this counterfactual experiment.

### Variants

- reviewed Registration ambiguity — all reviewed examples;
- reviewed Registration ambiguity with `p_print >= 0.50`;
- reviewed Registration ambiguity plus reviewed Printed structure;
- all distance-two candidates above `p_print` cut-offs 0.50, 0.75, 0.90 and 0.95;
- thin-continuous-proxy distance-two candidates above cut-offs 0.50, 0.75 and 0.90.

The reviewed-registration-all variant is the primary visual hypothesis. Generalised variants show what happens if the reviewed pattern is turned into an observable probability/morphology rule.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q "gradio>=5.0" opencv-python-headless pandas numpy pyarrow matplotlib openpyxl

## 1. Configuration

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, math, os, platform, sys, threading
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gradio as gr

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
V82D_ROOT=ROOT/'observed_print_classifier_v8_2d_ownership_review'
OUTPUT_ROOT=ROOT/'observed_print_classifier_v8_2e_forced_preservation'
SOURCE_V82D_RUN_OVERRIDE=None
SOURCE_V82C_RUN_OVERRIDE=None
SOURCE_V82_RUN_OVERRIDE=None

PRIMARY_VARIANT='reviewed_registration_all'
P_THRESHOLDS=[0.50,0.75,0.90,0.95]
THIN_THRESHOLDS=[0.50,0.75,0.90]
OVERLAY_ALPHA=0.62
REVIEW_RADIUS=26
CONTACT_SHEET_ROWS=10
MAX_COMPONENT_CROPS_PER_VARIANT=120
PUBLIC_LINK=True
REVIEWER_NAME='David Bolton'

RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_v8_2e_forced_preservation'
OUT=OUTPUT_ROOT/RUN
TABLES=OUT/'tables'; WHOLE=OUT/'whole_pages'; VISUAL=OUT/'visual_audit'; AUDIT=OUT/'audit'; REVIEW=OUT/'review'
for p in [OUT,TABLES,WHOLE,VISUAL,AUDIT,REVIEW]: p.mkdir(parents=True,exist_ok=False)
print('Output:',OUT)

## 2. Resolve the exact Notebook 33 → Notebook 32 → V8.2 lineage

In [ ]:
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()

def complete_runs(root,required):
    hits=[]
    if Path(root).exists():
        for p in Path(root).iterdir():
            if p.is_dir() and all((p/r).exists() for r in required): hits.append(p)
    return sorted(hits,key=lambda p:p.stat().st_mtime)

req33=['experiment_identity.json','tables/ownership_review.csv']
if SOURCE_V82D_RUN_OVERRIDE:
    V82D=Path(SOURCE_V82D_RUN_OVERRIDE)
else:
    hits=complete_runs(V82D_ROOT,req33)
    if not hits: raise RuntimeError(f'No complete ownership-review run below {V82D_ROOT}')
    V82D=hits[-1]
id33=json.loads((V82D/'experiment_identity.json').read_text())
if id33.get('version')!='8.2d-distance-two-ownership-review':
    raise RuntimeError(f"Unexpected Notebook 33 identity: {id33.get('version')}")

if SOURCE_V82C_RUN_OVERRIDE:
    V82C=Path(SOURCE_V82C_RUN_OVERRIDE)
else:
    V82C=Path(id33['source_notebook32_run'])
req32=['experiment_identity.json','tables/all_eligible_label_sensitivity.parquet']
missing=[r for r in req32 if not (V82C/r).exists()]
if missing: raise RuntimeError(f'Notebook 32 source incomplete: {missing}')
id32=json.loads((V82C/'experiment_identity.json').read_text())
if id32.get('version')!='8.2c-whole-population-label-sensitivity-audit':
    raise RuntimeError(f"Unexpected Notebook 32 identity: {id32.get('version')}")

if SOURCE_V82_RUN_OVERRIDE:
    V82=Path(SOURCE_V82_RUN_OVERRIDE)
else:
    V82=Path(id32['source_v82_run'])
req82=['experiment_identity.json','tables/all_eligible_oof_predictions.parquet','whole_pages']
missing=[r for r in req82 if not (V82/r).exists()]
if missing: raise RuntimeError(f'V8.2 source incomplete: {missing}')
id82=json.loads((V82/'experiment_identity.json').read_text())
if id82.get('version')!='8.2-development-whole-page' or id82.get('independent_data_used') is not False:
    raise RuntimeError('Notebook 34 requires the development-only V8.2 source run.')

ownership=pd.read_csv(V82D/'tables'/'ownership_review.csv').fillna('')
enriched=pd.read_parquet(V82C/'tables'/'all_eligible_label_sensitivity.parquet')
pred=pd.read_parquet(V82/'tables'/'all_eligible_oof_predictions.parquet')
for df in [ownership,enriched,pred]:
    df['page']=df.page.astype(str)
    for c in ['y','x']: df[c]=pd.to_numeric(df[c],errors='raise').astype(int)
    df['p_print']=pd.to_numeric(df.p_print,errors='raise').astype(float)
if ownership.duplicated(['page','y','x']).any() or enriched.duplicated(['page','y','x']).any() or pred.duplicated(['page','y','x']).any():
    raise RuntimeError('Duplicate page/y/x coordinates in a source table.')

# Fail closed if Notebook 32 is not an exact enrichment of the same V8.2 predictions.
check=enriched[['page','y','x','p_print']].merge(pred[['page','y','x','p_print']],on=['page','y','x'],suffixes=('_32','_82'),validate='one_to_one')
if len(check)!=len(pred) or not np.allclose(check.p_print_32,check.p_print_82,rtol=0,atol=1e-12):
    raise RuntimeError('Notebook 32 rows/probabilities do not match the V8.2 source run exactly.')

print('Notebook 33:',V82D)
print('Notebook 32:',V82C)
print('V8.2:',V82)
print('Ownership reviews:',len(ownership),'Eligible population:',len(enriched))

## 3. Declare and materialise forced-preservation variants

In [ ]:
required_enriched={'nearest_clean_ink_chebyshev','thin_continuous_proxy','decision','truth','source_group'}
missing=required_enriched-set(enriched.columns)
if missing: raise RuntimeError(f'Notebook 32 enriched table lacks {sorted(missing)}')

base=enriched.copy().reset_index(drop=True)
base['thin_continuous_proxy']=base.thin_continuous_proxy.astype(bool)
base_d2=base.nearest_clean_ink_chebyshev.eq(2)
reviewed=ownership[ownership.ownership_class.astype(str).str.len()>0].copy()
registration=reviewed[reviewed.ownership_class.eq('Registration ambiguity')].copy()
review_print=reviewed[reviewed.ownership_class.eq('Printed structure')].copy()
if registration.empty: raise RuntimeError('The ownership review contains no Registration ambiguity rows.')

variant_defs=[]
def add_variant(variant_id,label,rows,kind,threshold=None):
    q=rows.copy()
    q=q.drop_duplicates(['page','y','x'])
    q['variant_id']=variant_id; q['variant_label']=label; q['variant_kind']=kind; q['variant_threshold']=threshold
    variant_defs.append(q)

add_variant('reviewed_registration_all','Reviewed Registration ambiguity — all',registration,'reviewed_registration')
add_variant('reviewed_registration_p050','Reviewed Registration ambiguity — p_print >= 0.50',registration[registration.p_print>=.50],'reviewed_registration',.50)
add_variant('reviewed_registration_plus_print','Reviewed Registration ambiguity + reviewed Printed structure',pd.concat([registration,review_print],ignore_index=True),'reviewed_combined')
for t in P_THRESHOLDS:
    add_variant(f'all_d2_p{int(t*100):03d}',f'All distance-two — p_print >= {t:.2f}',base[base_d2&(base.p_print>=t)],'population_probability',t)
for t in THIN_THRESHOLDS:
    add_variant(f'thin_d2_p{int(t*100):03d}',f'Thin-proxy distance-two — p_print >= {t:.2f}',base[base_d2&base.thin_continuous_proxy&(base.p_print>=t)],'population_thin_probability',t)

forced=pd.concat(variant_defs,ignore_index=True,sort=False)
forced=forced[['variant_id','variant_label','variant_kind','variant_threshold','page','source_group','y','x','truth','p_print','decision','thin_continuous_proxy']]
forced['already_v82_print']=forced.decision.eq('PRINT')
forced['newly_forced']=~forced.already_v82_print
forced.to_parquet(TABLES/'forced_preservation_candidates.parquet',index=False)
forced.to_csv(TABLES/'forced_preservation_candidates.csv',index=False)

variant_manifest=(forced.groupby(['variant_id','variant_label','variant_kind','variant_threshold'],dropna=False,as_index=False).agg(
    selected_coordinates=('page','size'),newly_forced=('newly_forced','sum'),pages=('page','nunique'),source_groups=('source_group','nunique'),
    mean_p_print=('p_print','mean'),median_p_print=('p_print','median'),thin_proxy=('thin_continuous_proxy','sum')
))
variant_manifest.to_csv(TABLES/'variant_manifest.csv',index=False)
print(variant_manifest.to_string(index=False))

## 4. Render complete pages

For each affected page and variant:

- `v8_2_baseline.png` is the unchanged registered V8.2 page;
- `forced_preservation.png` starts from V8.2 and restores exact observed bytes at newly forced coordinates;
- `difference_from_v8_2.png` localises only the newly changed bytes;
- `forced_coordinates_overlay.png` marks forced coordinates on the observed page;
- `comparison.png` shows V8.2 and the counterfactual page at identical scale.

In [ ]:
CYAN=(255,255,0); MAGENTA=(255,0,255)
def overlay_points(gray,mask,colour=MAGENTA,alpha=OVERLAY_ALPHA,radius=1):
    active=mask.copy()
    if radius>0: active=cv2.dilate(active.astype(np.uint8),np.ones((2*radius+1,2*radius+1),np.uint8))>0
    base=cv2.cvtColor(gray,cv2.COLOR_GRAY2BGR); layer=base.copy(); layer[active]=colour
    out=base.copy(); out[active]=cv2.addWeighted(base,1-alpha,layer,alpha,0)[active]
    return out

def comparison_image(left,right,title_left,title_right):
    h,w=left.shape; banner=50
    canvas=np.full((h+banner,2*w,3),255,np.uint8)
    canvas[banner:,:w]=cv2.cvtColor(left,cv2.COLOR_GRAY2BGR); canvas[banner:,w:]=cv2.cvtColor(right,cv2.COLOR_GRAY2BGR)
    cv2.putText(canvas,title_left,(12,32),cv2.FONT_HERSHEY_SIMPLEX,.8,(30,30,30),2,cv2.LINE_AA)
    cv2.putText(canvas,title_right,(w+12,32),cv2.FONT_HERSHEY_SIMPLEX,.8,(30,30,30),2,cv2.LINE_AA)
    return canvas

page_rows=[]; integrity_rows=[]
for (variant_id,page),g in forced.groupby(['variant_id','page'],sort=False):
    src=V82/'whole_pages'/page
    required=['observed.png','v7_8.png','v8_2.png']
    miss=[n for n in required if not (src/n).exists()]
    if miss: raise RuntimeError(f'{page}: missing V8.2 pages {miss}')
    observed=cv2.imread(str(src/'observed.png'),0); v78=cv2.imread(str(src/'v7_8.png'),0); v82=cv2.imread(str(src/'v8_2.png'),0)
    if observed is None or v78 is None or v82 is None or not (observed.shape==v78.shape==v82.shape):
        raise RuntimeError(f'{page}: unreadable or mismatched V8.2 page set')
    ys=g.y.to_numpy(int); xs=g.x.to_numpy(int)
    if (ys<0).any() or (xs<0).any() or (ys>=observed.shape[0]).any() or (xs>=observed.shape[1]).any(): raise RuntimeError(f'{page}: forced coordinate outside page')
    selected=np.zeros(observed.shape,bool); selected[ys,xs]=True
    out=v82.copy(); out[selected]=observed[selected]
    changed=out!=v82
    # Integrity: every counterfactual difference must be selected and equal observed.
    outside_selection=int(np.sum(changed&~selected)); wrong=int(np.sum(changed&(out!=observed)))
    if outside_selection or wrong: raise RuntimeError(f'{variant_id}/{page}: integrity failure')
    d=WHOLE/variant_id/page; d.mkdir(parents=True,exist_ok=False)
    cv2.imwrite(str(d/'observed.png'),observed); cv2.imwrite(str(d/'v7_8.png'),v78); cv2.imwrite(str(d/'v8_2_baseline.png'),v82)
    cv2.imwrite(str(d/'forced_preservation.png'),out)
    diff=np.zeros_like(observed); diff[changed]=255; cv2.imwrite(str(d/'difference_from_v8_2.png'),diff)
    cv2.imwrite(str(d/'forced_coordinates_overlay.png'),overlay_points(observed,selected))
    cv2.imwrite(str(d/'comparison.png'),comparison_image(v82,out,f'{page} V8.2 baseline',f'{page} {variant_id}'))
    page_rows.append({'variant_id':variant_id,'page':page,'source_group':g.source_group.iloc[0],'selected_coordinates':int(selected.sum()),'changed_bytes':int(changed.sum()),'no_op_selected_bytes':int(selected.sum()-changed.sum()),'mean_p_print':float(g.p_print.mean()),'median_p_print':float(g.p_print.median())})
    integrity_rows.append({'variant_id':variant_id,'page':page,'outside_selection_changes':outside_selection,'changed_not_equal_observed':wrong,'forced_page_sha256':sha256_file(d/'forced_preservation.png'),'difference_sha256':sha256_file(d/'difference_from_v8_2.png')})

page_metrics=pd.DataFrame(page_rows); page_metrics.to_csv(TABLES/'variant_results_by_page.csv',index=False)
pd.DataFrame(integrity_rows).to_csv(AUDIT/'forced_page_integrity_and_hashes.csv',index=False)
source_metrics=page_metrics.groupby(['variant_id','source_group'],as_index=False).agg(pages=('page','nunique'),selected_coordinates=('selected_coordinates','sum'),changed_bytes=('changed_bytes','sum'),no_op_selected_bytes=('no_op_selected_bytes','sum'))
source_metrics.to_csv(TABLES/'variant_results_by_source_group.csv',index=False)
print('Rendered affected page/variant pairs:',len(page_metrics))

## 5. Counterfactual truth accounting — diagnostic only

In [ ]:
# These figures use the unchanged V8.2 truth and therefore do not adjudicate the registration hypothesis.
truth_rows=[]
for variant_id,g in forced[forced.newly_forced].groupby('variant_id'):
    truth_rows.append({'variant_id':variant_id,'newly_forced':len(g),'current_truth_print':int((g.truth==1).sum()),'current_truth_annotation':int((g.truth==0).sum()),'current_truth_uncertain':int((g.truth==2).sum()),'mean_p_print':float(g.p_print.mean()),'median_p_print':float(g.p_print.median())})
truth_accounting=pd.DataFrame(truth_rows).merge(variant_manifest[['variant_id','variant_label','variant_kind','variant_threshold']],on='variant_id',how='left')
truth_accounting.to_csv(TABLES/'counterfactual_truth_accounting.csv',index=False)
print(truth_accounting[['variant_id','newly_forced','current_truth_print','current_truth_annotation','current_truth_uncertain','median_p_print']].to_string(index=False))

## 6. Crop-level visual audit artefacts

In [ ]:
def crop_pad(img,y,x,r=REVIEW_RADIUS,pad=255):
    h,w=img.shape[:2]; y0,y1=y-r,y+r+1; x0,x1=x-r,x+r+1
    t=max(0,-y0); l=max(0,-x0); b=max(0,y1-h); rr=max(0,x1-w)
    q=img[max(0,y0):min(h,y1),max(0,x0):min(w,x1)]
    return cv2.copyMakeBorder(q,t,b,l,rr,cv2.BORDER_CONSTANT,value=pad)

def candidate_sheet(page,y,x,variant_id):
    d=WHOLE/variant_id/page
    names=['v7_8.png','v8_2_baseline.png','forced_preservation.png','difference_from_v8_2.png']
    titles=['V7.8','V8.2 baseline','Forced preservation','Difference']
    ims=[]
    for name in names:
        path=(V82/'whole_pages'/page/name) if name=='v7_8.png' else d/name
        im=cv2.imread(str(path),0); q=crop_pad(im,y,x)
        q=cv2.cvtColor(q,cv2.COLOR_GRAY2BGR); q=cv2.resize(q,(q.shape[1]*3,q.shape[0]*3),interpolation=cv2.INTER_NEAREST)
        cv2.drawMarker(q,(q.shape[1]//2,q.shape[0]//2),MAGENTA,cv2.MARKER_CROSS,17,2)
        ims.append(q)
    h,w=ims[0].shape[:2]; head=55; out=np.full((h+head,4*w,3),255,np.uint8)
    for i,(im,title) in enumerate(zip(ims,titles)):
        out[head:,i*w:(i+1)*w]=im; cv2.putText(out,title,(i*w+5,20),0,.45,(20,20,20),1,cv2.LINE_AA)
    cv2.putText(out,f'{page} x={x} y={y} {variant_id}',(5,44),0,.42,(20,20,20),1,cv2.LINE_AA)
    return out

crop_inventory=[]
for variant_id,g in forced[forced.newly_forced].groupby('variant_id'):
    q=g.sort_values('p_print',ascending=False).head(MAX_COMPONENT_CROPS_PER_VARIANT)
    cdir=VISUAL/variant_id/'crops'; cdir.mkdir(parents=True,exist_ok=True)
    files=[]
    for _,r in q.iterrows():
        name=f'{r.page}_y{int(r.y)}_x{int(r.x)}.png'; path=cdir/name
        cv2.imwrite(str(path),candidate_sheet(str(r.page),int(r.y),int(r.x),variant_id)); files.append(path)
        crop_inventory.append({'variant_id':variant_id,'page':r.page,'y':int(r.y),'x':int(r.x),'p_print':float(r.p_print),'file':str(path.relative_to(OUT))})
    for start in range(0,len(files),CONTACT_SHEET_ROWS):
        batch=[cv2.imread(str(p)) for p in files[start:start+CONTACT_SHEET_ROWS]]
        if not batch: continue
        width=max(i.shape[1] for i in batch); height=sum(i.shape[0] for i in batch); canvas=np.full((height,width,3),255,np.uint8); yy=0
        for im in batch: canvas[yy:yy+im.shape[0],:im.shape[1]]=im; yy+=im.shape[0]
        cv2.imwrite(str(VISUAL/variant_id/f'contact_sheet_{start//CONTACT_SHEET_ROWS+1:02d}.png'),canvas)
pd.DataFrame(crop_inventory).to_csv(AUDIT/'crop_inventory.csv',index=False)

## 7. Whole-page contact sheets

In [ ]:
def page_contact_sheet(variant_id,rows,max_pages=12):
    rows=rows.head(max_pages)
    if rows.empty:return
    fig,axes=plt.subplots(len(rows),2,figsize=(12,max(5,6*len(rows))),squeeze=False)
    for r,(_,row) in enumerate(rows.iterrows()):
        d=WHOLE/variant_id/row.page
        for c,(name,title) in enumerate([('v8_2_baseline.png',f'{row.page} V8.2'),('forced_preservation.png',f'{row.page} forced')]):
            im=cv2.imread(str(d/name),0); axes[r,c].imshow(im,cmap='gray',vmin=0,vmax=255); axes[r,c].set_title(title); axes[r,c].axis('off')
    plt.tight_layout(); dest=VISUAL/variant_id; dest.mkdir(parents=True,exist_ok=True); plt.savefig(dest/'whole_page_most_changed.png',dpi=130,bbox_inches='tight'); plt.close(fig)
for variant_id,g in page_metrics.groupby('variant_id'):
    page_contact_sheet(variant_id,g.sort_values(['changed_bytes','selected_coordinates'],ascending=False))
print('Visual audit:',VISUAL)

## 8. Interactive visual-impact reviewer

The primary reviewed-registration variant is reviewed at the exact candidate coordinates. Categories answer the practical counterfactual question:

- **Beneficial printed structure restored**
- **Annotation visibly retained**
- **No visible difference**
- **Mixed / uncertain visual effect**

This review does not modify ownership labels or V8.2 truth.

In [ ]:
PRIMARY_REVIEW_CSV=REVIEW/'primary_variant_visual_review.csv'
PRIMARY_REVIEW_XLSX=REVIEW/'primary_variant_visual_review.xlsx'
primary=forced[(forced.variant_id==PRIMARY_VARIANT)&forced.newly_forced].copy().sort_values(['page','y','x']).reset_index(drop=True)
primary['visual_effect']=''; primary['visual_note']=''; primary['reviewer']=''; primary['reviewed_utc']=''; primary['review_revision']=0
if PRIMARY_REVIEW_CSV.exists():
    old=pd.read_csv(PRIMARY_REVIEW_CSV).fillna(''); fields=['page','y','x','visual_effect','visual_note','reviewer','reviewed_utc','review_revision']
    primary=primary.drop(columns=fields[3:]).merge(old[fields],on=['page','y','x'],how='left',validate='one_to_one')
    for c in fields[3:-1]: primary[c]=primary[c].fillna('')
    primary.review_revision=pd.to_numeric(primary.review_revision,errors='coerce').fillna(0).astype(int)
VISUAL_EFFECTS=['Beneficial printed structure restored','Annotation visibly retained','No visible difference','Mixed / uncertain visual effect']
LOCK=threading.RLock()

def persist_review():
    tmp=PRIMARY_REVIEW_CSV.with_suffix('.csv.tmp'); primary.to_csv(tmp,index=False); os.replace(tmp,PRIMARY_REVIEW_CSV)
    tmp=PRIMARY_REVIEW_XLSX.with_name(PRIMARY_REVIEW_XLSX.stem+'.tmp.xlsx'); primary.to_excel(tmp,index=False,engine='openpyxl'); os.replace(tmp,PRIMARY_REVIEW_XLSX)
persist_review()

def ids(filter_name):
    if filter_name=='Unreviewed only': return primary.index[primary.visual_effect.astype(str).str.len().eq(0)].tolist()
    if filter_name=='Reviewed only': return primary.index[primary.visual_effect.astype(str).str.len().gt(0)].tolist()
    return primary.index.tolist()
def render(filter_name,pos):
    ii=ids(filter_name)
    if not ii:return np.full((400,1000,3),245,np.uint8),'No matching records','', '',0
    pos=max(0,min(int(pos),len(ii)-1)); r=primary.loc[ii[pos]]; im=candidate_sheet(str(r.page),int(r.y),int(r.x),PRIMARY_VARIANT)
    meta=f"**{pos+1}/{len(ii)}** — {r.page} ({int(r.x)}, {int(r.y)}) — p_print={float(r.p_print):.6f} — previous={r.visual_effect or 'unreviewed'}"
    done=int(primary.visual_effect.astype(str).str.len().gt(0).sum())
    return cv2.cvtColor(im,cv2.COLOR_BGR2RGB),meta,str(r.visual_effect),str(r.visual_note),pos
def move(f,p,d):
    ii=ids(f); return render(f,0 if not ii else (int(p)+d)%len(ii))
def save(f,p,effect,note):
    ii=ids(f)
    if not ii or effect not in VISUAL_EFFECTS:return render(f,p)
    idx=ii[max(0,min(int(p),len(ii)-1))]
    with LOCK:
        primary.at[idx,'visual_effect']=effect; primary.at[idx,'visual_note']=str(note or ''); primary.at[idx,'reviewer']=REVIEWER_NAME; primary.at[idx,'reviewed_utc']=datetime.now(timezone.utc).isoformat(); primary.at[idx,'review_revision']=int(primary.at[idx,'review_revision'])+1; persist_review()
    return render(f,p) if f=='Unreviewed only' else move(f,p,1)

def export_review(): persist_review(); return [str(PRIMARY_REVIEW_CSV),str(PRIMARY_REVIEW_XLSX)]

with gr.Blocks(title='Notebook 34 Forced-Preservation Visual Review') as demo:
    gr.Markdown('# Notebook 34 — Registration-Ambiguity Forced-Preservation Review')
    gr.Markdown('Judge the change from **V8.2 baseline** to **Forced preservation** at the purple crosshair. The distinction is whether exact observed-byte restoration repairs print, retains annotation, or is visually irrelevant.')
    with gr.Row(): filt=gr.Dropdown(['Unreviewed only','Reviewed only','All'],value='Unreviewed only',label='Filter'); refresh=gr.Button('Refresh')
    image=gr.Image(interactive=False,height=620); meta=gr.Markdown()
    effect=gr.Radio(VISUAL_EFFECTS,label='Visual effect'); note=gr.Textbox(label='Optional note',lines=2)
    with gr.Row(): prev=gr.Button('Previous'); save_btn=gr.Button('Save & Next',variant='primary'); nxt=gr.Button('Next'); export_btn=gr.Button('Export')
    pos=gr.State(0); files=gr.Files()
    outs=[image,meta,effect,note,pos]
    demo.load(lambda:render('Unreviewed only',0),outputs=outs); filt.change(lambda f:render(f,0),inputs=filt,outputs=outs); refresh.click(render,inputs=[filt,pos],outputs=outs)
    prev.click(lambda f,p:move(f,p,-1),inputs=[filt,pos],outputs=outs); nxt.click(lambda f,p:move(f,p,1),inputs=[filt,pos],outputs=outs)
    save_btn.click(save,inputs=[filt,pos,effect,note],outputs=outs); export_btn.click(export_review,outputs=files)
print('Primary visual-review rows:',len(primary))

## 9. Final identity and summary

In [ ]:
environment={'python':sys.version,'opencv':cv2.__version__,'numpy':np.__version__,'pandas':pd.__version__,'gradio':gr.__version__,'platform':platform.platform()}
identity={
 'created_utc':datetime.now(timezone.utc).isoformat(),'version':'8.2e-registration-ambiguity-forced-preservation-visual-audit',
 'development_only':True,'promotion_authority':False,'retrained':False,'refitted':False,'recalibrated':False,'truth_labels_changed':False,
 'source_v82d_run':str(V82D),'source_v82c_run':str(V82C),'source_v82_run':str(V82),
 'source_v82d_identity_sha256':sha256_file(V82D/'experiment_identity.json'),'ownership_review_sha256':sha256_file(V82D/'tables'/'ownership_review.csv'),
 'source_v82c_identity_sha256':sha256_file(V82C/'experiment_identity.json'),'source_v82_identity_sha256':sha256_file(V82/'experiment_identity.json'),
 'primary_variant':PRIMARY_VARIANT,'probability_thresholds':P_THRESHOLDS,'thin_probability_thresholds':THIN_THRESHOLDS,
 'variant_manifest_sha256':sha256_file(TABLES/'variant_manifest.csv'),'candidate_table_sha256':sha256_file(TABLES/'forced_preservation_candidates.parquet'),
 'software':environment
}
(OUT/'experiment_identity.json').write_text(json.dumps(identity,indent=2,default=str))
summary_lines=['V8.2e REGISTRATION-AMBIGUITY FORCED-PRESERVATION VISUAL-IMPACT AUDIT','',f'V8.2 source: {V82}',f'Notebook 32 source: {V82C}',f'Ownership review source: {V82D}','','Retrained: False','Refitted: False','Recalibrated: False','Truth labels changed: False','',f'Primary variant: {PRIMARY_VARIANT}',f'Reviewed registration-ambiguity coordinates: {len(registration)}','',variant_manifest.to_string(index=False),'','Counterfactual renderings are diagnostic only and do not replace registered V8.2 evidence.']
(OUT/'executive_summary.txt').write_text('\n'.join(summary_lines))
print('\n'.join(summary_lines))

## 10. Launch reviewer

In [ ]:
demo.queue(default_concurrency_limit=1).launch(share=PUBLIC_LINK,debug=True)

## Outputs to bring back

Start with:

- `executive_summary.txt`
- `experiment_identity.json`
- `tables/variant_manifest.csv`
- `tables/variant_results_by_page.csv`
- `tables/counterfactual_truth_accounting.csv`
- `review/primary_variant_visual_review.csv`
- the primary variant's `visual_audit/reviewed_registration_all/contact_sheet_*.png`
- representative `whole_pages/reviewed_registration_all/<page>/comparison.png` files.